# 04b — End-to-end cross-impact sLCC joint valuation + factorial sensitivity (current data)

This notebook regenerates the **3 × 2 BC2/BO2 joint-valuation map** and adds an end-to-end
**factorial sensitivity map** for the three societal valuation assumptions used in the sLCC framework.
It does **not** read the stale legacy 27-grid used by the previous `04_spatial_slcc_market_gap_visualization` notebook.

It recomputes the complete figure input chain from the current project model:

1. current state electricity and feedstock prices;
2. current 2025-USD damage-cost schedules;
3. current BC2/BO2 spatial TEA;
4. current BC2/BO2 spatial LCIA using the regenerated regional electricity/diesel/natural-gas factors;
5. current sLCC revaluation under three joint cases, with **SPC fixed at 1.10**;
6. the cross-impact metric used by the current sLCC implementation;
7. a **2³ factorial sensitivity analysis** over social discount rate, SPC, and damage-cost scenario; and
8. 3 × 2 U.S. choropleths showing the factorial main effect of increasing each sLCC assumption.

### Cross-impact metric
For each state and pathway,

\[
\Delta sLCC_{cross}(\%) = 100\left[\frac{1}{6}\sum_m\frac{sLCC_{green,m}}{sLCC_{market,m}}-1\right]
\]

where the average is taken over the six canonical LCIA categories. This is **not** the additive `_ALL_` damage row.

### Joint valuation cases shown
- `r = 1%, low damage cost, SPC = 1.05`
- `r = 3%, central damage cost, SPC = 1.10`
- `r = 5%, high damage cost, SPC = 1.20`

### Factorial sensitivity ranges
- social discount rate: `1%` vs `5%`
- SPC: `1.05` vs `1.20`
- damage-cost scenario: `low` vs `high`

For each factor, the plotted sensitivity is the **factorial main effect**:
the mean cross-impact sLCC response at the high level minus the mean response at the low level,
averaging over all combinations of the other two factors. Because the response is already expressed in percent,
the map values are **percentage-point changes**, not relative percent changes.

The market benchmark is **synthetic graphite**.


## 1. Project setup and figure configuration

The notebook is intended to live at the project root beside `graphite_sus/`. It uses the same project name, TEA workbook, state-price snapshot, LCIA methods, and sLCC functions as notebook 03.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "graphite_sus").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate a project root containing graphite_sus/. "
        f"Current working directory: {here}"
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphite_sus.spatial.config import SpatialPaths, DEFAULT_SPATIAL_METHODS, CANON_IMPACTS
from graphite_sus.spatial.prices import (
    load_state_electricity_prices,
    load_state_feedstock_prices,
    build_state_price_table,
)
from graphite_sus.spatial.damage_costs import DamageCostLibrary
from graphite_sus.market_reference import load_market_references

PROJECT_NAME = "graphite_sys_v3"
TEA_WORKBOOK = PROJECT_ROOT / "graphite_sus" / "data" / "test" / "meb_tea_char_coke_v53.xlsx"

PATHWAYS = ("s_c2", "s_o2")
PATHWAY_LABELS = {"s_c2": "BC2", "s_o2": "BO2"}
TEA_PROFILE = "literature_hybrid"

ELECTRICITY_SNAPSHOT = "2025-03-01"
ELECTRICITY_PRICE_MODE = "snapshot"

MARKET_KIND = "synthetic"
SPC_FIXED = 1.10
SPC_LOW = 1.05
SPC_CENTRAL = 1.10
SPC_HIGH = 1.20

JOINT_SCENARIOS = [
    {
        "social_rate": 0.01,
        "spc": 1.05,
        "damage_scenario": "low",
        "label": "r = 1%, low damage cost",
    },
    {
        "social_rate": 0.03,
        "spc": 1.10,
        "damage_scenario": "central",
        "label": "r = 3%, central damage cost",
    },
    {
        "social_rate": 0.05,
        "spc": 1.20,
        "damage_scenario": "high",
        "label": "r = 5%, high damage cost",
    },
]
COLORSCALE = "RdBu_r"
COLOR_SCALE_MODE = "row"   # reproduces the supplied/reference figure semantics

PATHS = SpatialPaths()
METHODS = DEFAULT_SPATIAL_METHODS

OUTPUT_DIR = PROJECT_ROOT / "result" / "spatial" / "joint_valuation_end_to_end"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEA workbook:", TEA_WORKBOOK)
print("Output directory:", OUTPUT_DIR)
print("Pathways:", PATHWAYS)
print("Market benchmark:", MARKET_KIND)
print("Joint valuation cases:")
for case in JOINT_SCENARIOS:
    print(
        f"  r={case['social_rate']:.0%}, "
        f"SPC={case['spc']:.2f}, "
        f"damage={case['damage_scenario']}"
    )


PROJECT_ROOT: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2
TEA workbook: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/graphite_sus/data/test/meb_tea_char_coke_v53.xlsx
Output directory: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end
Pathways: ('s_c2', 's_o2')
Market benchmark: synthetic
Joint valuation cases:
  r=1%, SPC=1.05, damage=low
  r=3%, SPC=1.10, damage=central
  r=5%, SPC=1.20, damage=high


## 2. Audit current package inputs

This cell deliberately rejects missing current regional inputs. In particular, the state-to-energy-source mapping may be external to the package, but it must be available for a genuinely end-to-end rerun.


In [2]:
input_audit = PATHS.audit()
display(input_audit)

required_inputs = set(input_audit["input"])
missing_inputs = input_audit.loc[~input_audit["exists"], ["input", "path"]]
if not missing_inputs.empty:
    raise FileNotFoundError(
        "End-to-end mode requires every current spatial input. Missing:\n"
        + missing_inputs.to_string(index=False)
    )

assert TEA_WORKBOOK.exists(), TEA_WORKBOOK

# Current state prices (same logic as notebook 03)
ele = load_state_electricity_prices(
    PATHS.electricity_prices,
    snapshot=ELECTRICITY_SNAPSHOT,
    mode=ELECTRICITY_PRICE_MODE,
)
feed = load_state_feedstock_prices(PATHS.feedstock_prices)
state_prices = build_state_price_table(ele, feed)

assert state_prices["state"].nunique() >= 48
print("State price coverage:", state_prices["state"].nunique(), "states/DC")
display(state_prices.head())

# Current 2025-USD damage-cost library (same source and rebasing as notebook 03)
damage_lib = DamageCostLibrary(
    PATHS.damage_reference,
    PATHS.cpi_reference,
    PATHS.damage_overrides,
    target_dollar_year=2025,
)
damage_reference = damage_lib.rebased_reference()
damage_schedules = damage_lib.schedules(years_ops=30)
damage_schedule_table = damage_lib.schedule_table(years_ops=30)

for cat in CANON_IMPACTS:
    lo = damage_schedules[cat]["low"]
    cen = damage_schedules[cat]["central"]
    hi = damage_schedules[cat]["high"]
    assert np.all(lo <= cen) and np.all(cen <= hi), cat

display(
    damage_reference[
        ["impact_category", "low_target", "central_target", "high_target", "schedule_mode", "annual_real_growth"]
    ]
)

market_refs = load_market_references()
display(market_refs)


,input,path,exists
0,electricity_prices,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
1,feedstock_prices,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
2,market_impacts,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
3,state_geometries,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
4,damage_reference,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
5,damage_overrides,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
6,cpi_reference,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
7,state_source_map,/mnt/g/My Drive/ElectricityLCI_derived_data/en...,True
8,electricity_ef,/mnt/g/My Drive/yale/Project-graphite/graphite...,True
9,diesel_ef,/mnt/g/My Drive/yale/Project-graphite/graphite...,True


State price coverage: 51 states/DC


,state,electricity_usd_per_kwh,feedstock_usd_per_kg
0,AK,0.2016,0.030294
1,AL,0.0744,0.066169
2,AR,0.0628,0.064973
3,AZ,0.0784,0.059991
4,CA,0.1984,0.062781


,impact_category,low_target,central_target,high_target,schedule_mode,annual_real_growth
0,climate change,0.143557,0.267595,0.391634,geometric,0.015
1,acidification,0.300966,0.518190,0.735415,constant,0.000
2,eutrophication,25.298778,31.435211,37.571644,constant,0.000
3,ecotoxicity: freshwater,0.002316,0.006123,0.009931,constant,0.000
4,particulate matter formation,692.599684,764.378197,836.156709,constant,0.000
5,photochemical oxidant formation,0.007556,0.307892,0.608228,constant,0.000


,market_type,quantity,lo,hi,mean,unit,geography,source,ref,note
0,natural,MSP,2.8,4.5,3.65,USD/kg,U.S.,project_selected_market_range,NaN,Authoritative project market-natural-graphite ...
1,synthetic,MSP,4.2,5.3,4.75,USD/kg,U.S.,project_selected_market_range,NaN,Authoritative project market-synthetic-graphit...


## 3. Brightway/system preflight and current TEA bundles

Notebook 01 remains the model-construction prerequisite: the Brightway project and `graphite_system` must already contain the regenerated central fuel factors. This cell checks that condition rather than silently rebuilding or using an older database.


In [3]:
import bw2data as bd

from graphite_sus.workflow import prepare_scenario, deterministic_streams, run_deterministic_tea
from graphite_sus.spatial.tea import SpatialTEARunner
from graphite_sus.fuel_reference import (
    audit_system_fuel_reference,
    audit_spatial_ef_ranges,
    fuel_reference_manifest,
)

bd.projects.set_current(PROJECT_NAME)

fuel_ef_audit = audit_system_fuel_reference("graphite_system")
if fuel_ef_audit.empty:
    raise RuntimeError(
        "No fuel UF parameters were found in graphite_system. "
        "Run notebook 01 using the current regenerated fuel reference first."
    )
display(fuel_ef_audit)
assert bool(fuel_ef_audit["passed"].all()), (
    "The Brightway graphite_system fuel UFs do not match the current "
    "graphite_sus/data/fuel_ef_netl_reference.csv. Rerun notebook 01."
)

# Only BC2 and BO2 are needed for this figure.
bundles = {
    sc: prepare_scenario(
        sc,
        workbook_path=TEA_WORKBOOK,
        electricity_mode="national_2025",
    )
    for sc in PATHWAYS
}
streams_by_scenario = {sc: deterministic_streams(b)[0] for sc, b in bundles.items()}
tea_runner = SpatialTEARunner(bundles, tea_profile=TEA_PROFILE)

# Reconcile the spatial TEA runner against deterministic TEA at identical prices.
recon = []
for sc, bundle in bundles.items():
    det, _, _ = run_deterministic_tea(bundle, tea_profile=TEA_PROFILE)
    base = tea_runner.baseline_prices(sc)
    spa, _ = tea_runner.run(
        sc,
        electricity_usd_per_kwh=base["electricity"],
        feedstock_usd_per_kg=base["feedstock"],
    )
    a = float(det.summary["msp_usd_per_kg"])
    z = float(spa.summary["msp_usd_per_kg"])
    recon.append({
        "scenario": sc,
        "deterministic_msp": a,
        "spatial_runner_at_same_prices": z,
        "abs_delta": abs(a - z),
        "pass": np.isclose(a, z, atol=1e-9, rtol=0),
    })

recon_df = pd.DataFrame(recon)
display(recon_df)
assert recon_df["pass"].all()


,activity,variable,actual,expected,passed
0,"graphite production, from coke, allocation (sy...",UF_IPCC_2021__climate_change__global_warming_p...,0.709453,0.709453,True
1,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__acidification__acidification_po...,0.005252,0.005252,True
2,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__eutrophication__eutrophication_...,0.000328,0.000328,True
3,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__ecotoxicity_freshwater__ecotoxi...,1.401552,1.401552,True
4,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__particulate_matter_formation__p...,0.000101,0.000101,True
5,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__photochemical_oxidant_formation...,0.182250,0.182250,True
6,"graphite production, from coke, allocation (sy...",UF_IPCC_2021__climate_change__global_warming_p...,0.785507,0.785507,True
7,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__acidification__acidification_po...,0.000866,0.000866,True
8,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__eutrophication__eutrophication_...,0.000055,0.000055,True
9,"graphite production, from coke, allocation (sy...",UF_TRACI_v2_1__ecotoxicity_freshwater__ecotoxi...,0.356956,0.356956,True


{'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Carbon_dioxide_fossil__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char_alloc', 'amount': 1.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Dinitrogen_monoxide__air__non_urban_air_or_from_high_stacks': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char_alloc', 'amount': 273.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Dinitrogen_monoxide__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char_alloc', 'amount': 273.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Methane_fossil__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char_alloc', 'amount': 29.8}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Nitrogen_oxides__air__non_urb

,scenario,deterministic_msp,spatial_runner_at_same_prices,abs_delta,pass
0,s_c2,4.399352,4.399352,0.0,True
1,s_o2,5.268109,5.268109,0.0,True


## 4. Recompute current BC2/BO2 state TEA

Only state electricity and feedstock prices are changed; all other TEA streams and assumptions remain identical to the deterministic model.


In [4]:
state_tea = tea_runner.table(state_prices, PATHWAYS)

assert state_tea["state"].nunique() == state_prices["state"].nunique()
assert set(state_tea["scenario"]) == set(PATHWAYS)
assert not state_tea["MSP"].isna().any()

display(state_tea.groupby("scenario")["MSP"].agg(["min", "mean", "max"]))


,min,mean,max
scenario,,,
s_c2,4.115199,4.429029,5.765265
s_o2,4.832228,5.227914,6.334275


## 5. Recompute current BC2/BO2 state LCIA

This is the same read-only spatial LCIA path as notebook 03: state-specific electricity, diesel, and natural-gas characterization factors are substituted into the existing symbolic mass-allocation formulas without mutating the Brightway database.


In [5]:
from graphite_sus.spatial.emission_factors import StateSourceMap, EFTables
from graphite_sus.spatial.lca import StateLCACalculator, state_lca_table, allocation_ratio_audit
from graphite_sus.validate import REFERENCE_MASS_FRACTION
from graphite_sus.electricity_regeneration import (
    audit_regenerated_table as audit_electricity_regenerated_table,
    audit_uncertainty_spec as audit_electricity_uncertainty_spec,
)

# Exact-method/provenance audit for the regenerated electricity table.
electricity_ef_audit = audit_electricity_regenerated_table(
    PATHS.electricity_ef,
    expected_methods=METHODS,
)
display(electricity_ef_audit)
assert bool(electricity_ef_audit["passed"].all()), (
    "Electricity EF table is missing/stale or lacks the exact current IPCC-AR6/TRACI-2.1 provenance."
)

# Keep the uncertainty envelope synchronized with the regenerated regional table.
electricity_uncertainty_audit = audit_electricity_uncertainty_spec()
display(electricity_uncertainty_audit)
assert bool(electricity_uncertainty_audit["passed"].all())

state_map = StateSourceMap.from_csv(PATHS.state_source_map)
ef_tables = EFTables.from_csvs(PATHS.electricity_ef, PATHS.diesel_ef, PATHS.natural_gas_ef)

fuel_spatial_range_audit = audit_spatial_ef_ranges(ef_tables)
display(fuel_spatial_range_audit)
assert bool(fuel_spatial_range_audit["passed"].all()), (
    "The spatial diesel/natural-gas factor envelopes no longer match the current central fuel reference."
)

# Independent mass-allocation diagnostic for the two pathways used here.
allocation_audit = allocation_ratio_audit(streams_by_scenario)
expected_by_sc = {
    "s_c2": REFERENCE_MASS_FRACTION["graphtz_char_alloc"],
    "s_o2": REFERENCE_MASS_FRACTION["graphtz_coke_alloc"],
}
allocation_audit["reference_fraction"] = allocation_audit["scenario"].map(expected_by_sc)
allocation_audit["abs_delta_from_reference"] = (
    allocation_audit["graphite_mass_fraction"] - allocation_audit["reference_fraction"]
).abs()
allocation_audit["pass"] = allocation_audit["abs_delta_from_reference"] < 0.02

display(allocation_audit)
assert allocation_audit["pass"].all()

states = sorted(set(state_prices["state"]) & set(state_map.by_state.index))
assert len(states) >= 48, f"Only {len(states)} states overlap price and source-map inputs"

calc = StateLCACalculator(state_map, ef_tables)
state_lca = state_lca_table(
    states=states,
    bundles=bundles,
    streams_by_scenario=streams_by_scenario,
    methods=METHODS,
    calculator=calc,
)

assert state_lca["state"].nunique() == len(states)
assert set(state_lca["scenario"]) == set(PATHWAYS)
assert not state_lca[list(CANON_IMPACTS)].isna().any().any()

display(state_lca.groupby("scenario")[list(CANON_IMPACTS)].agg(["min", "mean", "max"]))


,check,passed,detail
0,electricity_columns_exact,True,"location, climate change, acidification, eutro..."
1,electricity_locations_unique,True,8
2,electricity_regions_complete,True,missing=[]; extra=[]
3,electricity_values_finite_nonnegative,True,
4,electricity_methods_exact,True,"[('IPCC 2021', 'climate change', 'global warmi..."
5,electricity_climate_is_ipcc2021_ar6,True,IPCC 2021 AR6 GWP100
6,electricity_nonclimate_is_traci21,True,TRACI v2.1 (same registered Brightway methods ...
7,electricity_csv_hash_matches_metadata,True,89589e1502bc8196c0c11eca542f210926b787c7c2f755...


,impact_category,variable,n_rows,spec_lo,reference_lo,spec_hi,reference_hi,passed
0,climate change,UF_IPCC_2021__climate_change__global_warming_p...,1,0.232226,0.232226,0.894510,0.894510,True
1,acidification,UF_TRACI_v2_1__acidification__acidification_po...,1,0.000254,0.000254,0.005731,0.005731,True
2,eutrophication,UF_TRACI_v2_1__eutrophication__eutrophication_...,1,0.000120,0.000120,0.005144,0.005144,True
3,ecotoxicity: freshwater,UF_TRACI_v2_1__ecotoxicity_freshwater__ecotoxi...,1,1.934268,1.934268,6.984719,6.984719,True
4,particulate matter formation,UF_TRACI_v2_1__particulate_matter_formation__p...,1,0.000039,0.000039,0.002107,0.002107,True
5,photochemical oxidant formation,UF_TRACI_v2_1__photochemical_oxidant_formation...,1,0.006355,0.006355,0.064376,0.064376,True


,carrier,impact_category,n_regions,actual_low_per_MJ,reference_low_per_MJ,actual_high_per_MJ,reference_high_per_MJ,low_matches,high_matches,passed,method_provenance_status,errors
0,natural_gas,climate change,14,0.014213,0.014213,0.025421,0.025421,True,True,True,IPCC_AR6_GWP100_REGENERATED_FROM_NETL_ELEMENTA...,
1,natural_gas,acidification,14,0.000082,0.000082,0.000211,0.000211,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
2,natural_gas,eutrophication,14,0.000005,0.000005,0.000013,0.000013,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
3,natural_gas,ecotoxicity: freshwater,14,0.006261,0.006261,0.072038,0.072038,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
4,natural_gas,particulate matter formation,14,0.000002,0.000002,0.000004,0.000004,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
5,natural_gas,photochemical oxidant formation,14,0.002847,0.002847,0.007334,0.007334,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
6,diesel,climate change,5,0.016094,0.016094,0.020611,0.020611,True,True,True,IPCC_AR6_GWP100_REGENERATED_FROM_NETL_ELEMENTA...,
7,diesel,acidification,5,0.000016,0.000016,0.000025,0.000025,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
8,diesel,eutrophication,5,0.000001,0.000001,0.000002,0.000002,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,
9,diesel,ecotoxicity: freshwater,5,0.005842,0.005842,0.010838,0.010838,True,True,True,TRACI_2.1_FEDEFL_UUID_REGENERATED_FROM_NETL_EL...,


,scenario,graphite_mass_fraction,allocation_scheme,reference_fraction,abs_delta_from_reference,pass
0,s_c2,0.737027,mass,0.737,0.000027,True
1,s_o2,0.552984,mass,0.553,0.000016,True


climate change                     acidification                      \
                    min      mean       max           min      mean       max   
scenario                                                                        
s_c2           4.040685  5.077465  7.041655      0.011849  0.015993  0.036251   
s_o2           2.848850  3.480639  4.678437      0.009216  0.011744  0.024087   

         eutrophication                     ecotoxicity: freshwater            \
                    min      mean       max                     min      mean   
scenario                                                                        
s_c2           0.009434  0.018961  0.031844               63.433631  72.89845   
s_o2           0.008766  0.014572  0.022424               47.794923  53.56445   

                    particulate matter formation                      \
                max                          min      mean       max   
scenario                                                               
s_c2      86.792109                     0.002172  0.005596  0.011414   
s_o2      62.050212                     0.001715  0.003802  0.007348   

         photochemical oxidant formation                     
                                     min     mean       max  
scenario                                                     
s_c2                            0.218591  0.26111  0.476980  
s_o2                            0.166405  0.19240  0.323868

## 6. Revalue the current state model under the three joint societal scenarios

TEA is solved only once per state/pathway and then revalued under each societal-assumption case. This is equivalent to the current `state_slcc_table` logic but avoids recomputing identical private TEA cash flows three times.


In [6]:
from graphite_sus.spatial.slcc import (
    compare_state_to_market,
    impact_average_full_slcc_pct,
    market_impact_means,
    state_slcc_table,
)

# Use only the exact states present in the freshly recomputed LCIA table.
price_used = state_prices.loc[state_prices["state"].isin(states)].copy()
lidx = state_lca.set_index(["state", "scenario"])
market_df = pd.read_csv(PATHS.market_impacts)
market_intensities = market_impact_means(market_df, MARKET_KIND, CANON_IMPACTS)

rows = []
for r in price_used.itertuples(index=False):
    for sc in PATHWAYS:
        key = (r.state, sc)
        if key not in lidx.index:
            raise KeyError(f"Missing state LCIA row for {key}")

        # Cached by SpatialTEARunner: one private TEA solution per state/pathway.
        res, _ = tea_runner.run(
            sc,
            electricity_usd_per_kwh=float(r.electricity_usd_per_kwh),
            feedstock_usd_per_kg=float(r.feedstock_usd_per_kg),
        )

        li = lidx.loc[key]
        green_intensities = {cat: float(li[cat]) for cat in CANON_IMPACTS}

        for case in JOINT_SCENARIOS:
            cmp = compare_state_to_market(
                res=res,
                green_intensities=green_intensities,
                market_intensities=market_intensities,
                damage_schedules=damage_schedules,
                market_kind=MARKET_KIND,
                social_rate=float(case["social_rate"]),
                spc=float(case["spc"]),
                damage_scenario=str(case["damage_scenario"]),
            )
            cmp.insert(0, "scenario", sc)
            cmp.insert(0, "state", r.state)
            cmp["MSP_model"] = float(res.summary["msp_usd_per_kg"])
            cmp["electricity_price"] = float(r.electricity_usd_per_kwh)
            cmp["feedstock_price"] = float(r.feedstock_usd_per_kg)
            cmp["market_type"] = MARKET_KIND
            cmp["social_rate"] = float(case["social_rate"])
            cmp["spc"] = float(case["spc"])
            cmp["damage_scenario"] = str(case["damage_scenario"])
            cmp["valuation_case"] = str(case["label"])
            rows.append(cmp)

joint_slcc_raw = pd.concat(rows, ignore_index=True)
scenario_avg = impact_average_full_slcc_pct(joint_slcc_raw)

case_lookup = {
    (float(c["social_rate"]), float(c["spc"]), str(c["damage_scenario"])): str(c["label"])
    for c in JOINT_SCENARIOS
}
scenario_avg["valuation_case"] = [
    case_lookup[(float(sr), float(sp), str(ds))]
    for sr, sp, ds in scenario_avg[["social_rate", "spc", "damage_scenario"]].itertuples(index=False, name=None)
]

coverage = (
    scenario_avg.groupby(["valuation_case", "scenario"])["state"]
    .nunique()
    .rename("n_states")
    .reset_index()
)
display(coverage)

expected_n = len(states)
assert (coverage["n_states"] == expected_n).all(), coverage
assert len(scenario_avg) == expected_n * len(PATHWAYS) * len(JOINT_SCENARIOS)

summary = (
    scenario_avg.groupby(["valuation_case", "scenario"])["impact_avg_full_pct"]
    .agg(["min", "mean", "max"])
    .reset_index()
)
display(summary)


,valuation_case,scenario,n_states
0,"r = 1%, low damage cost",s_c2,51
1,"r = 1%, low damage cost",s_o2,51
2,"r = 3%, central damage cost",s_c2,51
3,"r = 3%, central damage cost",s_o2,51
4,"r = 5%, high damage cost",s_c2,51
5,"r = 5%, high damage cost",s_o2,51


,valuation_case,scenario,min,mean,max
0,"r = 1%, low damage cost",s_c2,-21.616423,-14.702656,14.764829
1,"r = 1%, low damage cost",s_o2,-24.172456,-16.229283,7.022916
2,"r = 3%, central damage cost",s_c2,-19.585044,-12.732283,17.058382
3,"r = 3%, central damage cost",s_o2,-20.466952,-12.694194,10.565901
4,"r = 5%, high damage cost",s_c2,-17.763580,-10.432537,19.629314
5,"r = 5%, high damage cost",s_o2,-15.398875,-7.759491,15.505172


## 7. Closure test against the current baseline sLCC implementation

The central row is recomputed once more through the public `state_slcc_table` function. The two paths must agree exactly on the cross-impact result. This replaces the old notebook's comparison against the stale `legacy_27grid.csv`.


In [7]:
central_baseline_raw = state_slcc_table(
    state_prices=price_used,
    scenarios=PATHWAYS,
    tea_runner=tea_runner,
    lca_table=state_lca,
    market_df=market_df,
    damage_schedules=damage_schedules,
    market_kind=MARKET_KIND,
    social_rate=0.03,
    spc=SPC_CENTRAL,
    damage_scenario="central",
)
central_baseline_avg = impact_average_full_slcc_pct(central_baseline_raw)

central_joint = scenario_avg.loc[
    np.isclose(scenario_avg["social_rate"], 0.03)
    & np.isclose(scenario_avg["spc"], SPC_CENTRAL)
    & scenario_avg["damage_scenario"].eq("central"),
    ["state", "scenario", "impact_avg_full_pct"],
].rename(columns={"impact_avg_full_pct": "joint_crossimpact_pct"})

central_reference = central_baseline_avg[
    ["state", "scenario", "impact_avg_full_pct"]
].rename(columns={"impact_avg_full_pct": "reference_crossimpact_pct"})

closure = central_joint.merge(
    central_reference,
    on=["state", "scenario"],
    how="inner",
    validate="one_to_one",
)
closure["abs_diff"] = (
    closure["joint_crossimpact_pct"] - closure["reference_crossimpact_pct"]
).abs()

max_diff = float(closure["abs_diff"].max())
print("Maximum central-case closure difference:", max_diff)
assert max_diff < 1e-10, closure.sort_values("abs_diff", ascending=False).head()
print("PASS: end-to-end joint-case calculation reproduces the current baseline implementation.")


Maximum central-case closure difference: 0.0
PASS: end-to-end joint-case calculation reproduces the current baseline implementation.


## 8. Factorial sensitivity of sLCC assumptions

The reference sensitivity figure is a **2³ factorial sensitivity analysis**, not the three joint
valuation cases above. Each of the three factors has a low and a high level, yielding eight
combinations per state and pathway.

For factor \(x\), the mapped main effect is

\[
E_x = \overline{\Delta sLCC_{cross}\mid x=x_{high}}
      - \overline{\Delta sLCC_{cross}\mid x=x_{low}},
\]

where the bar averages over the four combinations of the other two factors.

This makes the three rows directly interpretable as:
- social discount rate: 1% → 5%;
- SPC: 1.05 → 1.20;
- environmental damage costs: low → high.

The result is an **absolute change in the already-percent cross-impact metric**, so the unit is
**percentage points (pp)**.


In [8]:
from itertools import product

FACTORIAL_LEVELS = {
    "social_rate": (0.01, 0.05),
    "spc": (1.05, 1.20),
    "damage_scenario": ("low", "high"),
}

FACTORIAL_CASES = [
    {
        "social_rate": float(sr),
        "spc": float(sp),
        "damage_scenario": str(ds),
    }
    for sr, sp, ds in product(
        FACTORIAL_LEVELS["social_rate"],
        FACTORIAL_LEVELS["spc"],
        FACTORIAL_LEVELS["damage_scenario"],
    )
]
assert len(FACTORIAL_CASES) == 8

factorial_rows = []

for r in price_used.itertuples(index=False):
    for sc in PATHWAYS:
        key = (r.state, sc)
        if key not in lidx.index:
            raise KeyError(f"Missing state LCIA row for {key}")

        # SpatialTEARunner caches identical state/pathway TEA solves.
        res, _ = tea_runner.run(
            sc,
            electricity_usd_per_kwh=float(r.electricity_usd_per_kwh),
            feedstock_usd_per_kg=float(r.feedstock_usd_per_kg),
        )

        li = lidx.loc[key]
        green_intensities = {cat: float(li[cat]) for cat in CANON_IMPACTS}

        for case in FACTORIAL_CASES:
            cmp = compare_state_to_market(
                res=res,
                green_intensities=green_intensities,
                market_intensities=market_intensities,
                damage_schedules=damage_schedules,
                market_kind=MARKET_KIND,
                social_rate=case["social_rate"],
                spc=case["spc"],
                damage_scenario=case["damage_scenario"],
            )
            cmp.insert(0, "scenario", sc)
            cmp.insert(0, "state", r.state)
            cmp["market_type"] = MARKET_KIND
            cmp["social_rate"] = case["social_rate"]
            cmp["spc"] = case["spc"]
            cmp["damage_scenario"] = case["damage_scenario"]
            factorial_rows.append(cmp)

factorial_slcc_raw = pd.concat(factorial_rows, ignore_index=True)
factorial_avg = impact_average_full_slcc_pct(factorial_slcc_raw)

factorial_coverage = (
    factorial_avg.groupby(["social_rate", "spc", "damage_scenario", "scenario"])["state"]
    .nunique()
    .rename("n_states")
    .reset_index()
)

assert len(factorial_avg) == len(states) * len(PATHWAYS) * 8
assert (factorial_coverage["n_states"] == len(states)).all()
display(factorial_coverage)


,social_rate,spc,damage_scenario,scenario,n_states
0,0.01,1.05,high,s_c2,51
1,0.01,1.05,high,s_o2,51
2,0.01,1.05,low,s_c2,51
3,0.01,1.05,low,s_o2,51
4,0.01,1.20,high,s_c2,51
5,0.01,1.20,high,s_o2,51
6,0.01,1.20,low,s_c2,51
7,0.01,1.20,low,s_o2,51
8,0.05,1.05,high,s_c2,51
9,0.05,1.05,high,s_o2,51


In [9]:
EFFECT_SPECS = [
    {
        "effect": "social_rate",
        "factor": "social_rate",
        "low": 0.01,
        "high": 0.05,
        "title": "Effect of increasing social discount rate (1% → 5%)",
    },
    {
        "effect": "spc",
        "factor": "spc",
        "low": 1.05,
        "high": 1.20,
        "title": "Effect of increasing SPC factor (1.05 → 1.20)",
    },
    {
        "effect": "damage_cost",
        "factor": "damage_scenario",
        "low": "low",
        "high": "high",
        "title": "Effect of increasing damage costs (low → high)",
    },
]


def _select_level(df, factor, value):
    if factor in {"social_rate", "spc"}:
        return df.loc[np.isclose(pd.to_numeric(df[factor]), float(value))].copy()
    return df.loc[df[factor].astype(str).eq(str(value))].copy()


effect_parts = []

for spec in EFFECT_SPECS:
    factor = spec["factor"]

    # Balanced factorial main effect:
    # mean response at each factor level, averaging over all four combinations
    # of the other two factors.
    level_means = (
        factorial_avg
        .groupby(["state", "scenario", factor], as_index=False)["impact_avg_full_pct"]
        .mean()
    )

    low = _select_level(level_means, factor, spec["low"])[
        ["state", "scenario", "impact_avg_full_pct"]
    ].rename(columns={"impact_avg_full_pct": "response_low_pct"})

    high = _select_level(level_means, factor, spec["high"])[
        ["state", "scenario", "impact_avg_full_pct"]
    ].rename(columns={"impact_avg_full_pct": "response_high_pct"})

    eff = low.merge(
        high,
        on=["state", "scenario"],
        how="inner",
        validate="one_to_one",
    )
    eff["effect_pp"] = eff["response_high_pct"] - eff["response_low_pct"]
    eff["effect"] = spec["effect"]
    eff["factor"] = factor
    eff["low_level"] = str(spec["low"])
    eff["high_level"] = str(spec["high"])
    eff["title"] = spec["title"]
    effect_parts.append(eff)

factorial_effects = pd.concat(effect_parts, ignore_index=True)

effect_coverage = (
    factorial_effects.groupby(["effect", "scenario"])["state"]
    .nunique()
    .rename("n_states")
    .reset_index()
)
assert len(factorial_effects) == len(states) * len(PATHWAYS) * len(EFFECT_SPECS)
assert (effect_coverage["n_states"] == len(states)).all()

effect_summary = (
    factorial_effects
    .groupby(["effect", "scenario"])["effect_pp"]
    .agg(
        min="min",
        mean="mean",
        max="max",
        max_abs=lambda x: float(np.max(np.abs(np.asarray(x, dtype=float)))),
    )
    .reset_index()
)
effect_summary["pathway"] = effect_summary["scenario"].map(PATHWAY_LABELS)

display(effect_coverage)
display(effect_summary[["effect", "pathway", "min", "mean", "max", "max_abs"]])


,effect,scenario,n_states
0,damage_cost,s_c2,51
1,damage_cost,s_o2,51
2,social_rate,s_c2,51
3,social_rate,s_o2,51
4,spc,s_c2,51
5,spc,s_o2,51


,effect,pathway,min,mean,max,max_abs
0,damage_cost,BC2,-1.373551,1.310285,3.218385,3.218385
1,damage_cost,BO2,-2.751936,-0.913013,0.505624,2.751936
2,social_rate,BC2,2.391111,2.415025,2.486453,2.486453
3,social_rate,BO2,7.601293,7.626404,7.678318,7.678318
4,spc,BC2,0.550207,0.550207,0.550207,0.550207
5,spc,BO2,1.773779,1.773779,1.773779,1.773779


## 9. Plot the 3 × 2 sLCC-assumption sensitivity map

Each row uses one symmetric color scale shared by BC2 and BO2. This is important because the
maps show a signed effect: red means that increasing the assumption makes the average cross-impact
sLCC comparison more positive, while blue means that it makes the comparison more negative.

The map uses **percentage points** because it is `high-level ΔsLCC (%) − low-level ΔsLCC (%)`.


In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

effect_order = ["social_rate", "spc", "damage_cost"]
effect_title = {s["effect"]: s["title"] for s in EFFECT_SPECS}

# Symmetric row-specific scales make direction and magnitude visually comparable
# between BC2 and BO2 within each assumption.
effect_limits = {}
for effect in effect_order:
    vals = pd.to_numeric(
        factorial_effects.loc[factorial_effects["effect"].eq(effect), "effect_pp"],
        errors="coerce",
    )
    vals = vals[np.isfinite(vals)]
    lim = float(np.max(np.abs(vals)))
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    effect_limits[effect] = (-lim, lim)

print("Symmetric sensitivity color limits (percentage points):")
for effect in effect_order:
    print(f"  {effect}: {effect_limits[effect]}")

subplot_titles = (
    "(a) BC2", "(b) BO2",
    "(c) BC2", "(d) BO2",
    "(e) BC2", "(f) BO2",
)

sens_fig = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "choropleth"}, {"type": "choropleth"}],
        [{"type": "choropleth"}, {"type": "choropleth"}],
        [{"type": "choropleth"}, {"type": "choropleth"}],
    ],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.055,
    vertical_spacing=0.085,
)

COLORBAR_Y = {1: 0.835, 2: 0.500, 3: 0.165}
COLORBAR_X = {1: 0.475, 2: 1.015}

for row_i, effect in enumerate(effect_order, start=1):
    zmin, zmax = effect_limits[effect]

    for col_i, sc in enumerate(PATHWAYS, start=1):
        d = (
            factorial_effects.loc[
                factorial_effects["effect"].eq(effect)
                & factorial_effects["scenario"].eq(sc),
                ["state", "effect_pp", "response_low_pct", "response_high_pct"],
            ]
            .copy()
            .sort_values("state")
        )

        customdata = np.column_stack([
            d["state"].to_numpy(),
            d["response_low_pct"].to_numpy(),
            d["response_high_pct"].to_numpy(),
            d["effect_pp"].to_numpy(),
        ])

        sens_fig.add_trace(
            go.Choropleth(
                locations=d["state"],
                z=d["effect_pp"],
                locationmode="USA-states",
                colorscale=COLORSCALE,
                zmid=0,
                zmin=zmin,
                zmax=zmax,
                marker_line_color="rgba(60,60,60,0.55)",
                marker_line_width=0.45,
                showscale=True,
                colorbar=dict(
                    title=dict(text="p.p.<br>change", side="top"),
                    x=COLORBAR_X[col_i],
                    y=COLORBAR_Y[row_i],
                    len=0.22,
                    thickness=13,
                    outlinewidth=0,
                    ticks="outside",
                    tickformat=".1f",
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Low-level cross-impact ΔsLCC: %{customdata[1]:+.2f}%<br>"
                    "High-level cross-impact ΔsLCC: %{customdata[2]:+.2f}%<br>"
                    "<b>Main effect: %{customdata[3]:+.2f} pp</b>"
                    "<extra></extra>"
                ),
            ),
            row=row_i,
            col=col_i,
        )

sens_fig.update_geos(
    scope="usa",
    showlakes=False,
    showland=True,
    landcolor="white",
    bgcolor="rgba(0,0,0,0)",
)

ROW_TITLE_Y = {1: 0.985, 2: 0.650, 3: 0.315}

for row_i, effect in enumerate(effect_order, start=1):
    sens_fig.add_annotation(
        x=0.5,
        y=ROW_TITLE_Y[row_i],
        xref="paper",
        yref="paper",
        text=effect_title[effect],
        showarrow=False,
        font=dict(size=16),
        xanchor="center",
        yanchor="bottom",
    )

sens_fig.update_layout(
    width=1120,
    height=1110,
    margin=dict(l=10, r=100, t=95, b=10),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

for ann in sens_fig.layout.annotations[:6]:
    ann.font.size = 15

sens_fig.show()


Symmetric sensitivity color limits (percentage points):
  social_rate: (-7.678317898756756, 7.678317898756756)
  spc: (-1.7737792192103399, 1.7737792192103399)
  damage_cost: (-3.2183847082007606, 3.2183847082007606)


## 10. Save factorial-sensitivity outputs

The state-level main effects, all eight factorial endpoint responses, and the per-impact raw sLCC
results are saved separately from the three joint-valuation-case outputs.


In [11]:
SENS_STEM = "crossimpact_slcc_factorial_sensitivity_BC2_BO2_current_end_to_end"

SENS_HTML_PATH = OUTPUT_DIR / f"{SENS_STEM}.html"
SENS_PNG_PATH = OUTPUT_DIR / f"{SENS_STEM}.png"
SENS_EFFECT_CSV = OUTPUT_DIR / f"{SENS_STEM}_main_effects.csv"
SENS_ENDPOINT_CSV = OUTPUT_DIR / f"{SENS_STEM}_factorial_endpoints.csv"
SENS_RAW_CSV = OUTPUT_DIR / f"{SENS_STEM}_per_impact_raw.csv"
SENS_MANIFEST_PATH = OUTPUT_DIR / f"{SENS_STEM}_manifest.json"

sens_fig.write_html(SENS_HTML_PATH, include_plotlyjs="cdn")

sens_png_written = False
try:
    sens_fig.write_image(SENS_PNG_PATH, scale=2.0)
    sens_png_written = True
except Exception as exc:
    warnings.warn(f"Sensitivity PNG export skipped because Plotly/Kaleido export failed: {exc}")

factorial_effects.to_csv(SENS_EFFECT_CSV, index=False)
factorial_avg.to_csv(SENS_ENDPOINT_CSV, index=False)
factorial_slcc_raw.to_csv(SENS_RAW_CSV, index=False)

sens_manifest = {
    "project": PROJECT_NAME,
    "purpose": "end-to-end 2^3 factorial sensitivity of cross-impact sLCC assumptions",
    "pathways": list(PATHWAYS),
    "market_kind": MARKET_KIND,
    "factorial_levels": {
        "social_rate": list(FACTORIAL_LEVELS["social_rate"]),
        "spc": list(FACTORIAL_LEVELS["spc"]),
        "damage_scenario": list(FACTORIAL_LEVELS["damage_scenario"]),
    },
    "n_factorial_cases": len(FACTORIAL_CASES),
    "effect_definition": (
        "factorial main effect = mean cross-impact sLCC (%) at high factor level "
        "- mean cross-impact sLCC (%) at low factor level, averaging over all combinations "
        "of the other two factors"
    ),
    "effect_unit": "percentage points",
    "cross_impact_definition": (
        "100 * (mean over six canonical impacts of sLCC_green / sLCC_market - 1)"
    ),
    "canonical_impacts": list(CANON_IMPACTS),
    "state_count": len(states),
    "outputs": {
        "html": str(SENS_HTML_PATH),
        "png": str(SENS_PNG_PATH) if sens_png_written else None,
        "main_effects_csv": str(SENS_EFFECT_CSV),
        "factorial_endpoints_csv": str(SENS_ENDPOINT_CSV),
        "per_impact_raw_csv": str(SENS_RAW_CSV),
    },
}
SENS_MANIFEST_PATH.write_text(json.dumps(sens_manifest, indent=2))

print("Saved factorial sensitivity outputs:")
print("  HTML:", SENS_HTML_PATH)
print("  PNG :", SENS_PNG_PATH if sens_png_written else "not written (Kaleido unavailable/failed)")
print("  main effects:", SENS_EFFECT_CSV)
print("  factorial endpoints:", SENS_ENDPOINT_CSV)
print("  raw per-impact:", SENS_RAW_CSV)
print("  manifest:", SENS_MANIFEST_PATH)


/tmp/ipykernel_656695/2474315742.py:17: UserWarning:

Sensitivity PNG export skipped because Plotly/Kaleido export failed: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Saved factorial sensitivity outputs:
  HTML: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_factorial_sensitivity_BC2_BO2_current_end_to_end.html
  PNG : not written (Kaleido unavailable/failed)
  main effects: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_factorial_sensitivity_BC2_BO2_current_end_to_end_main_effects.csv
  factorial endpoints: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_factorial_sensitivity_BC2_BO2_current_end_to_end_factorial_endpoints.csv
  raw per-impact: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2

## 11. Reproduce the same 3 × 2 joint-valuation map layout

Within each valuation row, BC2 and BO2 share the same color range. The neutral reference is zero, the right panel carries the row colorbar, and the title/subplot/row-label positions follow the prior figure code.


In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

row_limits = {}

if COLOR_SCALE_MODE == "row":
    for case in JOINT_SCENARIOS:
        label = case["label"]
        vals = pd.to_numeric(
            scenario_avg.loc[scenario_avg["valuation_case"].eq(label), "impact_avg_full_pct"],
            errors="coerce",
        )
        vals = vals[np.isfinite(vals)]
        row_limits[label] = (float(vals.min()), float(vals.max()))

elif COLOR_SCALE_MODE == "global":
    vals = pd.to_numeric(scenario_avg["impact_avg_full_pct"], errors="coerce")
    vals = vals[np.isfinite(vals)]
    zmin, zmax = float(vals.min()), float(vals.max())
    for case in JOINT_SCENARIOS:
        row_limits[case["label"]] = (zmin, zmax)
else:
    raise ValueError("COLOR_SCALE_MODE must be 'row' or 'global'.")

print("Color limits:")
for key, value in row_limits.items():
    print(" ", key, value)

subplot_titles = (
    "(a) BC2", "(b) BO2",
    "(c) BC2", "(d) BO2",
    "(e) BC2", "(f) BO2",
)

fig = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "choropleth"}, {"type": "choropleth"}],
        [{"type": "choropleth"}, {"type": "choropleth"}],
        [{"type": "choropleth"}, {"type": "choropleth"}],
    ],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.015,
    vertical_spacing=0.075,
)

COLORBAR_Y = {1: 0.835, 2: 0.500, 3: 0.165}

for row_i, case in enumerate(JOINT_SCENARIOS, start=1):
    case_label = case["label"]
    zmin, zmax = row_limits[case_label]

    for col_i, sc in enumerate(PATHWAYS, start=1):
        d = (
            scenario_avg.loc[
                scenario_avg["valuation_case"].eq(case_label)
                & scenario_avg["scenario"].eq(sc),
                ["state", "impact_avg_full_pct"],
            ]
            .copy()
            .sort_values("state")
        )

        show_scale = col_i == 2

        fig.add_trace(
            go.Choropleth(
                locations=d["state"],
                z=d["impact_avg_full_pct"],
                locationmode="USA-states",
                colorscale=COLORSCALE,
                zmid=0,
                zmin=zmin,
                zmax=zmax,
                marker_line_color="rgba(80,80,80,0.45)",
                marker_line_width=0.45,
                showscale=show_scale,
                colorbar=(
                    dict(
                        title=dict(text="ΔsLCC<br>(%)", side="top"),
                        x=1.015,
                        y=COLORBAR_Y[row_i],
                        len=0.22,
                        thickness=14,
                        outlinewidth=0,
                        ticks="outside",
                        tickformat=".0f",
                    )
                    if show_scale
                    else None
                ),
                customdata=np.stack(
                    [d["state"].to_numpy(), d["impact_avg_full_pct"].to_numpy()],
                    axis=-1,
                ),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Cross-impact ΔsLCC: %{customdata[1]:+.2f}%"
                    "<extra></extra>"
                ),
            ),
            row=row_i,
            col=col_i,
        )

fig.update_geos(
    scope="usa",
    showlakes=False,
    showland=True,
    landcolor="white",
    bgcolor="rgba(0,0,0,0)",
)

ROW_TITLE_Y = {1: 0.985, 2: 0.650, 3: 0.315}

for row_i, case in enumerate(JOINT_SCENARIOS, start=1):
    row_title = (
        f"{case['label']}, "
        f"SPC = {case['spc']:.2f}"
    )

    fig.add_annotation(
        x=0.5,
        y=ROW_TITLE_Y[row_i],
        xref="paper",
        yref="paper",
        text=row_title,
        showarrow=False,
        font=dict(size=16),
        xanchor="center",
        yanchor="bottom",
    )

fig.update_layout(
    title=dict(
        text="",
        x=0.5,
        xanchor="center",
        y=0.995,
        yanchor="top",
        font=dict(size=19),
    ),
    width=1050,
    height=1080,
    margin=dict(l=15, r=90, t=90, b=15),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

for ann in fig.layout.annotations[:6]:
    ann.font.size = 15

fig.show()


Color limits:
  r = 1%, low damage cost (-24.17245599481862, 14.764828624417635)
  r = 3%, central damage cost (-20.466952388148307, 17.058381964396286)
  r = 5%, high damage cost (-17.763579600055714, 19.62931369452312)


## 12. Save joint-valuation figure inputs, outputs, and an auditable run manifest

The HTML and CSV outputs do not require Kaleido. PNG export is attempted and skipped with a warning if the environment lacks a compatible Kaleido installation.


In [13]:
def sha256_file(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


STEM = "crossimpact_slcc_joint_valuation_scenarios_BC2_BO2_current_end_to_end"
HTML_PATH = OUTPUT_DIR / f"{STEM}.html"
PNG_PATH = OUTPUT_DIR / f"{STEM}.png"
CSV_PATH = OUTPUT_DIR / f"{STEM}.csv"
RAW_PATH = OUTPUT_DIR / f"{STEM}_per_impact_raw.csv"
TEA_PATH = OUTPUT_DIR / f"{STEM}_state_tea.csv"
LCA_PATH = OUTPUT_DIR / f"{STEM}_state_lca.csv"
MANIFEST_PATH = OUTPUT_DIR / f"{STEM}_manifest.json"

fig.write_html(HTML_PATH, include_plotlyjs="cdn")

png_written = False
try:
    fig.write_image(PNG_PATH, scale=2.0)
    png_written = True
except Exception as exc:
    warnings.warn(f"PNG export skipped because Plotly/Kaleido export failed: {exc}")

scenario_avg.to_csv(CSV_PATH, index=False)
joint_slcc_raw.to_csv(RAW_PATH, index=False)
state_tea.to_csv(TEA_PATH, index=False)
state_lca.to_csv(LCA_PATH, index=False)

fingerprint_paths = {
    "tea_workbook": TEA_WORKBOOK,
    "electricity_prices": PATHS.electricity_prices,
    "feedstock_prices": PATHS.feedstock_prices,
    "market_impacts": PATHS.market_impacts,
    "market_reference": PROJECT_ROOT / "graphite_sus" / "data" / "market_reference.csv",
    "damage_reference": PATHS.damage_reference,
    "damage_overrides": PATHS.damage_overrides,
    "cpi_reference": PATHS.cpi_reference,
    "state_source_map": PATHS.state_source_map,
    "electricity_ef": PATHS.electricity_ef,
    "diesel_ef": PATHS.diesel_ef,
    "natural_gas_ef": PATHS.natural_gas_ef,
}

manifest = {
    "project": PROJECT_NAME,
    "purpose": "end-to-end current-data reproduction of the 3x2 cross-impact sLCC joint valuation figure",
    "pathways": list(PATHWAYS),
    "market_kind": MARKET_KIND,
    "market_price_usd_per_kg": float(
        market_refs.loc[market_refs["market_type"].eq(MARKET_KIND), "mean"].iloc[0]
    ),
    "spc_central": SPC_CENTRAL,
    "joint_spc_levels": [
        float(case["spc"]) for case in JOINT_SCENARIOS
    ],
    "joint_scenarios": JOINT_SCENARIOS,
    "cross_impact_definition": "100 * (mean over six canonical impacts of sLCC_green / sLCC_market - 1)",
    "canonical_impacts": list(CANON_IMPACTS),
    "electricity_snapshot": ELECTRICITY_SNAPSHOT,
    "electricity_price_mode": ELECTRICITY_PRICE_MODE,
    "tea_profile": TEA_PROFILE,
    "damage_target_dollar_year": 2025,
    "allocation_scheme": "mass",
    "spatial_lca_mode": "read_only_formula_evaluation",
    "state_count": len(states),
    "central_closure_max_abs_pct_point": max_diff,
    "fuel_reference": fuel_reference_manifest(),
    "input_sha256": {
        name: sha256_file(path)
        for name, path in fingerprint_paths.items()
        if Path(path).exists() and Path(path).is_file()
    },
    "outputs": {
        "html": str(HTML_PATH),
        "png": str(PNG_PATH) if png_written else None,
        "crossimpact_csv": str(CSV_PATH),
        "per_impact_raw_csv": str(RAW_PATH),
        "state_tea_csv": str(TEA_PATH),
        "state_lca_csv": str(LCA_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

print("Saved:")
print("  HTML:", HTML_PATH)
print("  PNG :", PNG_PATH if png_written else "not written (Kaleido unavailable/failed)")
print("  CSV :", CSV_PATH)
print("  RAW :", RAW_PATH)
print("  TEA :", TEA_PATH)
print("  LCA :", LCA_PATH)
print("  manifest:", MANIFEST_PATH)


/tmp/ipykernel_656695/2368891923.py:26: UserWarning:

PNG export skipped because Plotly/Kaleido export failed: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




Saved:
  HTML: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_joint_valuation_scenarios_BC2_BO2_current_end_to_end.html
  PNG : not written (Kaleido unavailable/failed)
  CSV : /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_joint_valuation_scenarios_BC2_BO2_current_end_to_end.csv
  RAW : /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/spatial/joint_valuation_end_to_end/crossimpact_slcc_joint_valuation_scenarios_BC2_BO2_current_end_to_end_per_impact_raw.csv
  TEA : /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/resu

## 13. Final acceptance checks

A successful run means the figure was produced only after current inputs, state TEA, state LCIA, fuel/electricity provenance, state coverage, and the central-case sLCC closure all passed.


In [14]:
acceptance = pd.DataFrame([
    {"check": "all spatial inputs exist", "pass": missing_inputs.empty},
    {"check": "state price coverage", "pass": state_prices["state"].nunique() >= 48},
    {"check": "fuel UF reference matches current model", "pass": bool(fuel_ef_audit["passed"].all())},
    {"check": "electricity EF provenance", "pass": bool(electricity_ef_audit["passed"].all())},
    {"check": "spatial fuel ranges match central reference", "pass": bool(fuel_spatial_range_audit["passed"].all())},
    {"check": "allocation diagnostic", "pass": bool(allocation_audit["pass"].all())},
    {"check": "state LCIA complete", "pass": not state_lca[list(CANON_IMPACTS)].isna().any().any()},
    {"check": "three valuation cases x two pathways cover all states", "pass": bool((coverage["n_states"] == len(states)).all())},
    {"check": "factorial 2^3 cases cover all states/pathways", "pass": bool((factorial_coverage["n_states"] == len(states)).all())},
    {"check": "three factorial main effects cover all states/pathways", "pass": bool((effect_coverage["n_states"] == len(states)).all())},
    {"check": "central sLCC closure", "pass": max_diff < 1e-10},
])

display(acceptance)
assert acceptance["pass"].all(), acceptance
print("END-TO-END FIGURE WORKFLOW ACCEPTED")


,check,pass
0,all spatial inputs exist,True
1,state price coverage,True
2,fuel UF reference matches current model,True
3,electricity EF provenance,True
4,spatial fuel ranges match central reference,True
5,allocation diagnostic,True
6,state LCIA complete,True
7,three valuation cases x two pathways cover all...,True
8,factorial 2^3 cases cover all states/pathways,True
9,three factorial main effects cover all states/...,True


END-TO-END FIGURE WORKFLOW ACCEPTED
